##  Day24__Feature Engineering & sklearn Pipelines

## Part A — encoding and scaling categorical & numeric columns

## TASK 01 — One-hot, ordinal, and target encoding side by side

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import html

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


# 1. Load the dataset
df = pd.read_csv("telecom_churn.csv")

# Clean column names in case they contain formatting characters
df.columns = [
    html.unescape(str(col)).strip().strip("*").strip()
    for col in df.columns
]

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)


# 2. Separate features and target
X = df.drop(columns=["churn", "customer_id"]).copy()
y = df["churn"].copy()

# Convert Yes/No target labels to 1/0 if needed
if y.dtype == "object":
    y_clean = y.astype(str).str.strip().str.lower()

    if set(y_clean.unique()) <= {"yes", "no"}:
        y = y_clean.map({"no": 0, "yes": 1})
    else:
        raise ValueError("Check the values in the churn column.")


# 3. Identify the categorical and date columns
categorical_cols = [
    "contract_type",
    "payment_method",
    "region"
]

date_cols = [
    col for col in X.columns
    if "date" in col.lower()
]

print("\nCategorical columns:", categorical_cols)
print("Date columns:", date_cols)


# 4. Convert the date into numeric features
for col in date_cols:
    dates = pd.to_datetime(X[col], errors="coerce")

    X[f"{col}_year"] = dates.dt.year
    X[f"{col}_month"] = dates.dt.month
    X[f"{col}_day"] = dates.dt.day

X = X.drop(columns=date_cols)

# Ensure numeric columns are actually numeric
for col in X.columns:
    if col not in categorical_cols:
        X[col] = pd.to_numeric(X[col], errors="coerce")

numeric_cols = [
    col for col in X.columns
    if col not in categorical_cols
]

print("Numeric columns:", numeric_cols)


# 5. Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))


# 6. Create a target mean encoder
class TargetMeanEncoder(BaseEstimator, TransformerMixin):

    def fit(self, X, y):
        X = pd.DataFrame(X).reset_index(drop=True)
        y = pd.Series(np.asarray(y)).reset_index(drop=True)

        self.global_mean_ = y.mean()
        self.category_means_ = {}
        self.columns_ = list(X.columns)

        for col in self.columns_:
            temp = pd.DataFrame({
                "category": X[col].astype(str),
                "target": y
            })

            self.category_means_[col] = (
                temp.groupby("category")["target"].mean().to_dict()
            )

        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        X.columns = self.columns_

        encoded = pd.DataFrame(index=X.index)

        for col in self.columns_:
            encoded[col] = (
                X[col]
                .astype(str)
                .map(self.category_means_[col])
                .fillna(self.global_mean_)
            )

        return encoded.to_numpy()


# 7. Define the three encoders
encoders = {
    "One-Hot": OneHotEncoder(handle_unknown="ignore"),

    "Ordinal": OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ),

    "Target Mean": TargetMeanEncoder()
}


# 8. Compare the three encoding methods
results = []

for encoding_name, encoder in encoders.items():

    preprocessing = ColumnTransformer([
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_cols
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", encoder)
            ]),
            categorical_cols
        )
    ])
    
    model = Pipeline([
        ("preprocessing", preprocessing),
        ("classifier", LogisticRegression(max_iter=2000))
    ])

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy"
    )

    results.append({
        "Encoding": encoding_name,
        "Mean CV Accuracy": scores.mean(),
        "Standard Deviation": scores.std() })

    print(f"\n{encoding_name} Encoding")
    print("Fold scores:", np.round(scores, 4))
    print("Mean CV accuracy:", round(scores.mean(), 4))
    print("Standard deviation:", round(scores.std(), 4))


# 9. Display the comparison table
results_df = pd.DataFrame(results)

print("\nFinal Results:")
print(results_df.to_string(index=False))

# 10. Plot the results
plt.figure(figsize=(8, 5))

plt.bar(
    results_df["Encoding"],
    results_df["Mean CV Accuracy"]
)

plt.xlabel("Encoding Method")
plt.ylabel("Mean Cross-Validation Accuracy")
plt.title("One-Hot vs Ordinal vs Target Mean Encoding")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## TASK 02 — Scaling: standard, min-max, and robust

In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# 1. Load the Telecom Churn dataset
df = pd.read_csv("telecom_churn.csv")

# Standardize column names for easier handling
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

# Support common column-name variations
if "monthlycharges" in df.columns:
    df.rename(columns={"monthlycharges": "monthly_charges"}, inplace=True)

if "totalcharges" in df.columns:
    df.rename(columns={"totalcharges": "total_charges"}, inplace=True)

# 2. Clean the target column
if "churn" not in df.columns:
    raise ValueError(f"Churn column not found. Available columns: {df.columns.tolist()}")

df = df.dropna(subset=["churn"]).copy()

# Remove customer ID because it is not a useful predictive feature
if "customerid" in df.columns:
    df.drop(columns=["customerid"], inplace=True)

# Convert target labels to 0 and 1
y = df["churn"].astype(str).str.strip().str.lower()

if set(y.unique()) <= {"yes", "no"}:
    y = y.map({"no": 0, "yes": 1})
elif set(y.unique()) <= {"0", "1"}:
    y = y.astype(int)
else:
    raise ValueError(f"Unexpected Churn labels: {y.unique()}")

X = df.drop(columns=["churn"]).copy()

# Convert charge columns to numeric.
# Invalid or blank values become NaN and will be imputed.
for col in ["monthly_charges", "total_charges"]:
    if col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce")

# Convert other numeric-looking object columns when possible
for col in X.select_dtypes(include="object").columns:
    converted = pd.to_numeric(X[col], errors="coerce")

    if converted.notna().sum() > 0 and converted.notna().sum() >= X[col].notna().sum() * 0.9:
        X[col] = converted

# 3. Identify numeric and categorical features
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

# 4. Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 5. Examine possible outliers using the IQR method
for col in ["monthly_charges", "total_charges"]:
    if col in X_train.columns:
        values = X_train[col].dropna()

        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1

        lower_limit = q1 - 1.5 * iqr
        upper_limit = q3 + 1.5 * iqr

        outliers = values[
            (values < lower_limit) | (values > upper_limit)
        ]

        print(f"\n{col}")
        print("Minimum:", values.min())
        print("Maximum:", values.max())
        print("IQR lower limit:", round(lower_limit, 2))
        print("IQR upper limit:", round(upper_limit, 2))
        print("Number of potential outliers:", len(outliers))
        print("Some potential outlier values:", outliers.head(10).tolist())

# 6. Compare three scalers using the same Logistic Regression model
scalers = {
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "RobustScaler": RobustScaler()
}

results = []

for scaler_name, scaler in scalers.items():

    # Numeric features: impute missing values and scale
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", scaler)
    ])

    # Categorical features: impute missing values and encode
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    # Apply the appropriate preprocessing to each feature type
    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ])

    # Same model for a fair comparison
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("logistic_regression", LogisticRegression(max_iter=2000))
    ])

    # Five-fold cross-validation on training data
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy"
    )

    results.append({
        "Scaler": scaler_name,
        "Mean CV Accuracy": scores.mean(),
        "Standard Deviation": scores.std()
    })

# 7. Display comparison
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    by="Mean CV Accuracy",
    ascending=False
).reset_index(drop=True)

print("\nCross-validation comparison:")
print(results_df.round(4).to_string(index=False))

print("\nBest scaler:", results_df.loc[0, "Scaler"])
print(
    "Best mean CV accuracy:",
    round(results_df.loc[0, "Mean CV Accuracy"], 4)
)

C:\Users\KULDEEP\AppData\Local\Temp\ipykernel_5520\2417834839.py:57: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include="object").columns:


Numeric features: ['tenure_months', 'age', 'monthly_charges', 'total_charges']
Categorical features: ['customer_id', 'contract_type', 'payment_method', 'region', 'signup_date']

monthly_charges
Minimum: 18.0
Maximum: 150.0
IQR lower limit: -3.44
IQR upper limit: 131.89
Number of potential outliers: 10
Some potential outlier values: [150.0, 145.84, 141.13, 135.65, 137.47, 143.15, 132.39, 132.55, 133.2, 138.98]

total_charges
Minimum: 17.73
Maximum: 8337.18
IQR lower limit: -2146.35
IQR upper limit: 4411.61
Number of potential outliers: 95
Some potential outlier values: [5288.11, 4710.65, 5493.58, 7776.43, 4761.56, 5289.32, 5464.34, 4510.1, 5491.13, 6368.75]

Cross-validation comparison:
        Scaler  Mean CV Accuracy  Standard Deviation
  MinMaxScaler            0.7644              0.0098
StandardScaler            0.7613              0.0121
  RobustScaler            0.7613              0.0121

Best scaler: MinMaxScaler
Best mean CV accuracy: 0.7644


## Part B — creating features: polynomial, interaction, and dates

## TASK 03 — Polynomial and interaction features that earn their place

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, RidgeClassifier

# 1. Load the Telecom Churn dataset
df = pd.read_csv("telecom_churn.csv")

# Standardize column names
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

# Handle common column-name variations
df.rename(columns={
    "monthlycharges": "monthly_charges",
    "totalcharges": "total_charges",
    "customerid": "customer_id"
}, inplace=True)

# 2. Prepare the target column
if "churn" not in df.columns:
    raise ValueError(f"Churn column not found. Available columns: {df.columns.tolist()}")

df = df.dropna(subset=["churn"]).copy()

# Remove customer ID because it is not a useful predictive feature
if "customer_id" in df.columns:
    df.drop(columns=["customer_id"], inplace=True)

y = df["churn"].astype(str).str.strip().str.lower()

if set(y.unique()) <= {"yes", "no"}:
    y = y.map({"no": 0, "yes": 1})
elif set(y.unique()) <= {"0", "1"}:
    y = y.astype(int)
else:
    raise ValueError(f"Unexpected Churn labels: {y.unique()}")

X = df.drop(columns=["churn"]).copy()

# Convert charge columns to numeric
for col in ["monthly_charges", "total_charges"]:
    if col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce")

# Convert other numeric-looking columns where appropriate
for col in X.select_dtypes(include="object").columns:
    converted = pd.to_numeric(X[col], errors="coerce")

    if (
        converted.notna().sum() > 0
        and converted.notna().sum() >= X[col].notna().sum() * 0.9
    ):
        X[col] = converted

# 3. Identify numeric and categorical columns
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

# Select a sensible subset for polynomial expansion
poly_features = [
    col for col in ["monthly_charges", "total_charges"]
    if col in numeric_features
]

# Remaining numeric features will not be expanded
other_numeric_features = [
    col for col in numeric_features
    if col not in poly_features
]

if not poly_features:
    raise ValueError("Neither monthly_charges nor total_charges was found.")

print("Polynomial features:", poly_features)
print("Other numeric features:", other_numeric_features)
print("Categorical features:", categorical_features)

# 4. Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 5. Create a reusable preprocessing and model pipeline
def create_pipeline(use_polynomial=False, model=None):

    if model is None:
        model = LogisticRegression(max_iter=2000)

    # Preprocess selected numeric features
    if use_polynomial:
        numeric_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("polynomial", PolynomialFeatures(
                degree=2,
                include_bias=False
            )),
            ("scaler", StandardScaler())
        ])
    else:
        numeric_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])

    transformers = [
        ("selected_numeric", numeric_pipeline, poly_features)
    ]

    # Keep remaining numeric features unchanged apart from imputation/scaling
    if other_numeric_features:
        other_numeric_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])

        transformers.append((
            "other_numeric",
            other_numeric_pipeline,
            other_numeric_features
        ))

    # Encode categorical features
    if categorical_features:
        categorical_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ])

        transformers.append((
            "categorical",
            categorical_pipeline,
            categorical_features
        ))

    preprocessor = ColumnTransformer(transformers)

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

# 6. Compare Logistic Regression with and without polynomial features
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

baseline_model = create_pipeline(
    use_polynomial=False,
    model=LogisticRegression(max_iter=2000)
)

polynomial_model = create_pipeline(
    use_polynomial=True,
    model=LogisticRegression(max_iter=2000)
)

baseline_scores = cross_val_score(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy"
)

polynomial_scores = cross_val_score(
    polynomial_model,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy"
)

# 7. Display the comparison table
comparison = pd.DataFrame({
    "Model": [
        "Without Polynomial Features",
        "With Degree-2 Polynomial Features"
    ],
    "Mean CV Accuracy": [
        baseline_scores.mean(),
        polynomial_scores.mean()
    ],
    "CV Standard Deviation": [
        baseline_scores.std(),
        polynomial_scores.std()
    ]
})

print("\nPOLYNOMIAL FEATURE COMPARISON")
print(comparison.round(4).to_string(index=False))

difference = polynomial_scores.mean() - baseline_scores.mean()

print(f"\nAccuracy change: {difference:+.4f}")
print(f"Accuracy change in percentage points: {difference * 100:+.2f}")

if difference > 0:
    print("Polynomial features improved the mean CV accuracy.")
elif difference < 0:
    print("Polynomial features reduced the mean CV accuracy.")
else:
    print("Both approaches achieved the same mean CV accuracy.")

# 8. Identify the strongest polynomial interaction
polynomial_model.fit(X_train, y_train)

preprocessor = polynomial_model.named_steps["preprocessor"]
logistic_model = polynomial_model.named_steps["model"]

# Get names of the polynomial features
poly_transformer = (
    preprocessor.named_transformers_["selected_numeric"]
    .named_steps["polynomial"]
)

poly_names = poly_transformer.get_feature_names_out(poly_features)

# Polynomial features come first in the ColumnTransformer
poly_coefficients = logistic_model.coef_[0][:len(poly_names)]

interaction_results = pd.DataFrame({
    "Feature": poly_names,
    "Coefficient": poly_coefficients
})

# Select pairwise interactions, excluding squared terms
interaction_results = interaction_results[
    interaction_results["Feature"].str.contains(" ", regex=False)
].copy()

if not interaction_results.empty:
    interaction_results["Absolute Coefficient"] = (
        interaction_results["Coefficient"].abs()
    )

    interaction_results = interaction_results.sort_values(
        "Absolute Coefficient",
        ascending=False
    )

    print("\nPOLYNOMIAL INTERACTIONS")
    print(interaction_results.round(4).to_string(index=False))

    strongest = interaction_results.iloc[0]

    print(
        "\nStrongest interaction by absolute coefficient:",
        strongest["Feature"],
        "| Coefficient:",
        round(strongest["Coefficient"], 4)
    )
else:
    print("\nNo pairwise polynomial interactions were available.")

# 9. Compare Ridge-style regularization on the better feature setup
use_poly_for_ridge = difference > 0

ridge_model = create_pipeline(
    use_polynomial=use_poly_for_ridge,
    model=RidgeClassifier(alpha=1.0)
)

ridge_scores = cross_val_score(
    ridge_model,
    X_train,
    y_train,
    cv=cv,
    scoring="accuracy"
)

ridge_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Ridge Classifier"
    ],
    "Features": [
        "Polynomial" if use_poly_for_ridge else "Original",
        "Polynomial" if use_poly_for_ridge else "Original"
    ],
    "Mean CV Accuracy": [
        max(baseline_scores.mean(), polynomial_scores.mean()),
        ridge_scores.mean()
    ],
    "CV Standard Deviation": [
        min(
            baseline_scores.std()
            if baseline_scores.mean() >= polynomial_scores.mean()
            else polynomial_scores.std(),
            1.0
        ),
        ridge_scores.std()
    ]
})

print("\nREGULARIZATION COMPARISON")
print(ridge_comparison.round(4).to_string(index=False))

print(
    "\nRidge accuracy change compared with the better Logistic Regression model:",
    f"{ridge_scores.mean() - max(baseline_scores.mean(), polynomial_scores.mean()):+.4f}"
)

**1. What does `interaction_only=True` do, and when would you prefer it?**  
It creates interaction features between different variables but excludes powers such as \(x^2\). Prefer it when you want to capture relationships between features without adding squared terms.

**2. Why can polynomial features make LinearRegression numerically unstable, and how does Ridge help?**  
Polynomial features can be highly correlated, causing multicollinearity and unstable coefficient estimates. Ridge adds an L2 penalty to shrink coefficients, improving numerical stability.

**3. How do you choose the degree without overfitting?**  
Compare different polynomial degrees using cross-validation and select the degree with the best validation performance. Keep a separate test set for final evaluation.

## TASK 04 — Turn the date column into usable features

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# 1. Load your CSV file
df = pd.read_csv("telecom_churn.csv")

# Clean column names
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

# Convert signup_date to datetime
df["signup_date"] = pd.to_datetime(
    df["signup_date"], errors="coerce"
)

# 2. Confirm date information
print("Year:\n", df["signup_date"].dt.year.head())
print("Month:\n", df["signup_date"].dt.month.head())
print("Day of week:\n", df["signup_date"].dt.dayofweek.head())
print("Quarter:\n", df["signup_date"].dt.quarter.head())

# 3. Create numeric date features
latest_date = df["signup_date"].max()

df["signup_year"] = df["signup_date"].dt.year
df["signup_month"] = df["signup_date"].dt.month
df["signup_dayofweek"] = df["signup_date"].dt.dayofweek
df["signup_quarter"] = df["signup_date"].dt.quarter
df["days_since"] = (latest_date - df["signup_date"]).dt.days

# Remove rows without a target value
df = df.dropna(subset=["churn"]).copy()

# Convert churn labels to 0 and 1
y = df["churn"].astype(str).str.strip().str.lower()

if set(y.unique()) <= {"yes", "no"}:
    y = y.map({"no": 0, "yes": 1})
elif set(y.unique()) <= {"0", "1"}:
    y = y.astype(int)
else:
    raise ValueError("Check the values in the churn column.")

# Remove customer ID if present
drop_columns = ["churn", "signup_date"]

if "customerid" in df.columns:
    drop_columns.append("customerid")

# Baseline: exclude raw date and engineered date features
X_baseline = df.drop(
    columns=drop_columns + [
        "signup_year", "signup_month",
        "signup_dayofweek", "signup_quarter",
        "days_since"
    ],
    errors="ignore"
).copy()

# Improved dataset: include engineered date features
X_date = df.drop(columns=drop_columns).copy()

# Convert common numeric columns safely
for X in [X_baseline, X_date]:
    for col in ["monthly_charges", "total_charges"]:
        if col in X.columns:
            X[col] = pd.to_numeric(X[col], errors="coerce")

# 4. Build a reusable model
def evaluate_model(X, y):
    numeric_features = X.select_dtypes(
        include=["number"]
    ).columns.tolist()

    categorical_features = X.select_dtypes(
        exclude=["number"]
    ).columns.tolist()

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=2000))
    ])

    cv = StratifiedKFold(
        n_splits=5, shuffle=True, random_state=42
    )

    scores = cross_val_score(
        model, X, y, cv=cv, scoring="accuracy"
    )

    return scores.mean(), scores.std()


# 5. Compare cross-validation scores
baseline_mean, baseline_std = evaluate_model(
    X_baseline, y
)

date_mean, date_std = evaluate_model(
    X_date, y
)

print(f"Baseline accuracy: {baseline_mean:.4f} ± {baseline_std:.4f}")
print(f"With date features: {date_mean:.4f} ± {date_std:.4f}")
print(f"Improvement: {(date_mean - baseline_mean) * 100:.2f} percentage points")

if date_mean > baseline_mean:
    print("Date features improved cross-validation accuracy.")
elif date_mean < baseline_mean:
    print("Date features reduced cross-validation accuracy.")
else:
    print("Date features made no difference to cross-validation accuracy.")

## Part C — custom transformers, leakage, and the deliverable pipeline

## TASK 05 — Spot the leakage in this code, then fix in

1. Name the leak

StandardScaler.fit_transform(X) calculates the mean and standard deviation using all rows, including the test rows. This allows information from the test data to influence training preprocessing.

2. Consequence

The training transformation indirectly knows the distribution of unseen test data, including its mean and standard deviation. In real deployment, future data would not be available when fitting the scaler. This leakage can make evaluation less reliable, although it does not necessarily increase accuracy.

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Load your CSV
df = pd.read_csv("telecom_churn.csv")
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# Prepare target
df = df.dropna(subset=["churn"]).copy()
y = df["churn"].astype(str).str.strip().str.lower()
y = y.map({"yes": 1, "no": 0})  if set(y.unique()) <= {"yes", "no"} else y.astype(int)

# Engineer date features if signup_date exists
if "signup_date" in df.columns:
    dates = pd.to_datetime(df.pop("signup_date"), errors="coerce")
    df["signup_year"] = dates.dt.year
    df["signup_month"] = dates.dt.month
    df["signup_dayofweek"] = dates.dt.dayofweek
    df["signup_quarter"] = dates.dt.quarter

X = df.drop(columns=["churn", "customerid"], errors="ignore")

for col in ["monthly_charges", "total_charges"]:
    if col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce")

numeric = X.select_dtypes(include="number").columns.tolist()
categorical = X.select_dtypes(exclude="number").columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical)
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# LEAKY: fit preprocessing on all rows before cross-validation
X_leaky = preprocessor.fit_transform(X)
leaky_model = LogisticRegression(max_iter=2000)
leaky_score = cross_val_score(
    leaky_model, X_leaky, y, cv=cv, scoring="accuracy"
).mean()

# FIXED: preprocessing is fitted separately inside each training fold
fixed_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000))
])

fixed_score = cross_val_score(
    fixed_pipeline, X, y, cv=cv, scoring="accuracy"
).mean()

print(f"Leaky CV accuracy: {leaky_score:.4f}")
print(f"Fixed CV accuracy: {fixed_score:.4f}")
print(f"Difference: {(leaky_score - fixed_score) * 100:.2f} percentage points")

4. Explain the result

- Leaky version: The imputer, scaler, and encoder learn statistics or categories from all rows before cross-validation.
- Fixed version: The Pipeline learns preprocessing parameters only from each training fold.
- Score gap: The leaky score may be higher or lower; a difference is not guaranteed. Report the actual scores from your run. The fixed version provides a more trustworthy evaluation because validation-fold information does not influence preprocessing.
Important: The comparison isolates preprocessing leakage, but the leaky score is not a fair estimate of real-world performance.

**1. How does Pipeline prevent leakage?**  
A `Pipeline` with `cross_val_score` fits transformers only on each training fold and then applies them to the validation fold. This prevents validation data from influencing preprocessing.

**2. Where else can leakage occur?**
- **Target encoding:** Using target labels from validation data.
- **Imputation:** Calculating missing-value statistics using the entire dataset.
- **Feature selection:** Selecting features before splitting or cross-validation.
- **Scaling:** Calculating mean and standard deviation using all rows.

**3. What would you audit first?**  
I would check whether data is split before preprocessing, whether transformers are fitted only on training data, whether target information leaks into features, and whether future information is used to predict past outcomes.

##  TASK 06 — Assemble the full ColumnTransformer + Pipeline

In [ ]:
import pandas as pd
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# 1. Load dataset
df = pd.read_csv("telecom_churn.csv")
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# 2. Prepare target
df = df.dropna(subset=["churn"]).copy()
y = df["churn"].astype(str).str.strip().str.lower()

if set(y.unique()) <= {"yes", "no"}:
    y = y.map({"no": 0, "yes": 1})
elif set(y.unique()) <= {"0", "1"}:
    y = y.astype(int)
else:
    raise ValueError("Check the churn labels.")

# 3. Extract date features
if "signup_date" in df.columns:
    dates = pd.to_datetime(df.pop("signup_date"), errors="coerce")
    df["signup_year"] = dates.dt.year
    df["signup_month"] = dates.dt.month
    df["signup_dayofweek"] = dates.dt.dayofweek
    df["signup_quarter"] = dates.dt.quarter
    df["days_since"] = (dates.max() - dates).dt.days

# 4. Prepare features
X = df.drop(columns=["churn", "customerid"], errors="ignore")

for col in ["monthly_charges", "total_charges"]:
    if col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce")

numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()

# 5. Build preprocessing transformer
prep = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_cols),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_cols)
])

# 6. Build complete pipeline
pipe = Pipeline([
    ("prep", prep),
    ("clf", LogisticRegression(max_iter=2000))
])

# 7. Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy")

print("Fold accuracies:", scores)
print(f"Mean CV accuracy: {scores.mean():.4f}")
print(f"Standard deviation: {scores.std():.4f}")


### 1. What happens to columns left out of `ColumnTransformer`?
By default, `remainder='drop'`, so any column not assigned to a transformer is **dropped from the output**. [scikit-learn](https://sklearn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html?utm_source=chatgpt.com)

### 2. Why should we drop the ID column instead of encoding it?
A `customer_id` is an identifier, not a meaningful customer characteristic. Encoding it can create unnecessary features and cause overfitting without helping predictions.

### 3. How does `Pipeline` prevent leakage?
When used with `cross_val_score`, the Pipeline fits preprocessing steps only on each training fold and applies them to its validation fold. This prevents validation data from influencing preprocessing. [scikit-learn.org](https://scikit-learn.org/stable/modules/compose?utm_source=chatgpt.com)

## TASK 07 — Write your own custom transformer and plug it in

In [ ]:
import pandas as pd
import numpy as np

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression


# 1. Custom transformer
class OutlierCap(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)

        # Learn limits from training data only
        self.lower_ = np.nanpercentile(X, 25, axis=0)
        self.upper_ = np.nanpercentile(X, 75, axis=0)

        iqr = self.upper_ - self.lower_
        self.lower_ = self.lower_ - 1.5 * iqr
        self.upper_ = self.upper_ + 1.5 * iqr

        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)

        # Apply learned limits
        return np.clip(X, self.lower_, self.upper_)


# 2. Load dataset
df = pd.read_csv("telecom_churn.csv")
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

df = df.dropna(subset=["churn"]).copy()

# 3. Prepare target
y = df["churn"].astype(str).str.strip().str.lower()

if set(y.unique()) <= {"yes", "no"}:
    y = y.map({"no": 0, "yes": 1})
elif set(y.unique()) <= {"0", "1"}:
    y = y.astype(int)
else:
    raise ValueError("Check the churn labels.")

# 4. Prepare features
X = df.drop(columns=["churn", "customerid"], errors="ignore")

# Convert numeric-looking columns
for col in ["monthly_charges", "total_charges"]:
    if col in X.columns:
        X[col] = pd.to_numeric(X[col], errors="coerce")

# Convert signup_date into numeric date features
if "signup_date" in X.columns:
    dates = pd.to_datetime(X.pop("signup_date"), errors="coerce")
    X["signup_year"] = dates.dt.year
    X["signup_month"] = dates.dt.month
    X["signup_dayofweek"] = dates.dt.dayofweek

numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()

# 5. Preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("outlier_cap", OutlierCap()),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

prep = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

# 6. Complete model pipeline
pipe = Pipeline([
    ("prep", prep),
    ("clf", LogisticRegression(max_iter=2000))
])

# 7. Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    pipe, X, y, cv=cv, scoring="accuracy"
)

print("Fold accuracies:", scores)
print(f"Mean CV accuracy: {scores.mean():.4f}")
print(f"Standard deviation: {scores.std():.4f}")

### 1. What must `fit()` and `transform()` return?
- **`fit()`** learns parameters from training data and returns `self`.
- **`transform()`** returns the transformed data.
- They must be separate so the transformer learns only from training data, preventing data leakage.

### 2. When do we need `inverse_transform()`?
When we want to convert transformed data back to its original representation, such as reversing scaling to recover original feature values.

### 3. How does `cross_val_score()` prevent leakage?
It splits data into folds, calls `fit()` on each training fold, and uses the fitted Pipeline to transform and evaluate the held-out fold. This ensures preprocessing learns only from training data.